<a href="https://colab.research.google.com/github/lazyclaw-137/awesome-OpenClaw-Money-Maker/blob/main/graphcast_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌦️ GraphCast Weather Edge for Prediction Markets

## Setup
1. Open in Google Colab (T4 GPU or better)
2. Run all cells
3. Get temperature predictions before markets update

## Why GraphCast?
- **Speed:** <1 minute vs hours for NOAA
- **Accuracy:** 99.7% troposphere accuracy  
- **Edge:** Predict temperature before markets update

## Requirements
- Colab GPU runtime (T4 minimum, A100 optimal)
- ~16GB VRAM for 1° model, ~60GB VRAM for 0.25° model

In [1]:
# Cell 1: Check GPU
!nvidia-smi

Wed Mar  4 01:26:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
# Cell 2: Reset and Clean Install
%cd /content
!rm -rf graphcast
!git clone https://github.com/google-deepmind/graphcast.git
%cd graphcast
!pip install -q .
!pip install -q gcsfs
# Move back to root to avoid circular imports
%cd /content

/content
Cloning into 'graphcast'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 230 (delta 92), reused 83 (delta 83), pack-reused 114 (from 1)
Receiving objects: 100% (230/230), 1.68 MiB | 4.60 MiB/s, done.
Resolving deltas: 100% (128/128), done.
/content/graphcast
  Preparing metadata (setup.py) ... done
/content


In [22]:
# Cell 3: Find and Download checkpoint
import os
import gcsfs

os.makedirs("checkpoints", exist_ok=True)
local_path = "checkpoints/graphcast_params.npz"
fs = gcsfs.GCSFileSystem(token='anon')

print("🔍 Searching for available checkpoints in dm_graphcast bucket...")
try:
    # List all files in the params directory recursively
    available_files = fs.find("dm_graphcast/params/")
    available_params = [f for f in available_files if f.endswith('.npz')]

    if not available_params:
        print("❌ No .npz files found in the bucket.")
    else:
        print(f"Found {len(available_params)} potential checkpoints.")
        # Filter for operational 0.25 degree model
        gcs_path = next((p for p in available_params if "operational" in p and "0.25" in p), available_params[0])
        print(f"⏳ Downloading: {gcs_path}...")
        fs.get(gcs_path, local_path)
        print(f"✅ Downloaded to {local_path}")
except Exception as e:
    print(f"❌ Error accessing GCS: {e}")

🔍 Searching for available checkpoints in dm_graphcast bucket...
Found 3 potential checkpoints.
⏳ Downloading: dm_graphcast/params/GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz...
✅ Downloaded to checkpoints/graphcast_params.npz


In [23]:
# Cell 4: Load model
import os
import sys
import jax
import haiku as hk

# Ensure the installed package is found first
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast

model_path = "checkpoints/graphcast_params.npz"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Checkpoint not found at {model_path}. Please run Cell 3.")

with open(model_path, 'rb') as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)

params = ckpt.params
state = {}
print("✅ Model loaded successfully")
print(f"Model description: {ckpt.description}")

✅ Model loaded successfully
Model description: 
GraphCast model at 0.25deg resolution, with 13 pressure levels. This model is
trained on ERA5 data from 1979 to 2017, and fine-tuned on HRES-fc0 data from
2016 to 2021 and can be causally evaluated on 2022 and later years. This model
does not take `total_precipitation_6hr` as inputs and can make predictions in an
operational setting (i.e., initialised from HRES-fc0).



In [24]:
# Cell 5: Get weather prediction
# Download latest weather data from ERA5 or GFS
# Run inference
# Extract temperature for target location
print("🚧 Implement weather data fetch + inference")

🚧 Implement weather data fetch + inference


## Output
Temperature anomaly prediction for target date/location.
Compare to Polymarket/Kalshi prices for trading edge.

# Task
Fetch the latest ERA5 initial conditions using the CDS API, preprocess the data into the format required by GraphCast, and run the model inference using the checkpoint located at "checkpoints/graphcast_params.npz" to generate a 10-day weather forecast and extract the temperature prediction for a specific target location.

## Configure CDS API

### Subtask:
Set up the Copernicus Climate Data Store (CDS) API credentials to enable data fetching.


**Reasoning**:
I will install the `cdsapi` library and configure the required credentials in the home directory to enable weather data fetching from the Copernicus Climate Data Store.



In [27]:
import os

# 1. Install the cdsapi library
!pip install -q cdsapi

# 2. Create the .cdsapirc file in the home directory
# Note: Users must replace <UID> and <API_KEY> with their actual CDS credentials
cds_url = "https://cds.climate.copernicus.eu/api/v2"
cds_key = "YOUR_UID:YOUR_API_KEY"

with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f'url: {cds_url}\nkey: {cds_key}')

# 3. Verify file existence and set permissions
print(f"Config file created at: {os.path.expanduser('~/.cdsapirc')}")
os.chmod(os.path.expanduser('~/.cdsapirc'), 0o600)

# 4. Import and initialize client
import cdsapi
try:
    client = cdsapi.Client()
    print("✅ CDS API Client successfully initialized.")
except Exception as e:
    print(f"⚠️ Initialization note: {e} (Expected if keys are placeholders)")

Config file created at: /root/.cdsapirc
✅ CDS API Client successfully initialized.


## Fetch Initial Conditions

### Subtask:
Download the required atmospheric and surface variables from ERA5 for the current timestamp using the CDS API.


**Reasoning**:
I will provide the code to download the necessary surface and pressure level data from the Copernicus Climate Data Store (CDS) using the API, targeting the specific variables and pressure levels required by the GraphCast model.



In [28]:
import cdsapi
import datetime

c = cdsapi.Client()

# Define target time (approx 12 hours ago to ensure ERA5 availability)
target_datetime = datetime.datetime.utcnow() - datetime.timedelta(hours=12)
date_str = target_datetime.strftime('%Y-%m-%d')
time_str = f"{target_datetime.hour:02d}:00"

print(f"Fetching data for {date_str} at {time_str}...")

# 1. Download Surface Variables
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'format': 'netcdf',
        'variable': [
            '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind',
            'mean_sea_level_pressure', 'total_precipitation'
        ],
        'date': date_str,
        'time': time_str,
    },
    'surface.nc')

# 2. Download Pressure Level Variables (13 levels as per model description)
c.retrieve(
    'reanalysis-era5-pressure-levels',
    {
        'product_type': 'reanalysis',
        'format': 'netcdf',
        'variable': [
            'geopotential', 'temperature', 'specific_humidity',
            'u_component_of_wind', 'v_component_of_wind'
        ],
        'pressure_level': [
            '100', '150', '200', '250', '300', '400', '500', '600', '700', '850', '925', '1000', '50'
        ],
        'date': date_str,
        'time': time_str,
    },
    'pressure.nc')

print("✅ Surface and Pressure data downloaded successfully.")

/tmp/ipykernel_353/3787392869.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_datetime = datetime.datetime.utcnow() - datetime.timedelta(hours=12)


Fetching data for 2026-03-03 at 13:00...


2026-03-04 01:36:57,910 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels
INFO:cdsapi:Sending request to https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels


Exception: Exception: Climate Data Store API endpoint not found

It seems you are trying to access an API endpoint that does not exist.

It is possible that you are pointing to an outdated API endpoint which has been removed/renamed. 

Your api client version and/or configuration file may need to be updated.

For users of the standard cdsapi Python library, please proceed as follows:
- update your cdsapi client to the latest version available
- update your .cdsapirc file as expected ("url" should be set to https://cds.climate.copernicus.eu/api and "key" should be set to your API key WITHOUT the deprecated `<UID>:` prefix)

For users of the ecmwf-datastores-client Python library (recommended for advanced users), please ensure that your ecmwf-datastores-client version is updated and your .ecmwfdatastoresrc file is correctly configured.

If you are trying to access a REST API endpoint directly, please refer to the documentation for more information on how to access REST API endpoints: htt